In [2]:
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import os

C:\Users\priyanshu\AppData\Local\Temp\ipykernel_27940\1314614971.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


In [3]:
# Load cleaned IKEA dataset

df = pd.read_csv("../data/cleaned_ikea.csv")

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (1200, 14)
     item_id    name  category  price  \
0  IKEA00001    ALEX     Chair   2138   
1  IKEA00002  KALLAX     Chair  14827   
2  IKEA00003    LACK      Sofa  10962   
3  IKEA00004   BESTA  Wardrobe  40064   
4  IKEA00005    IVAR  Wardrobe  38336   

                                  short_description          designer  depth  \
0        Space-saving furniture with easy assembly.     Henrik Preutz     51   
1  Compact furniture suitable for small apartments.      Ola Wihlborg     97   
2        Space-saving furniture with easy assembly.     Monika Mulder     63   
3   Durable furniture with practical storage space.  IKEA Design Team     78   
4  Compact furniture suitable for small apartments.  IKEA Design Team     25   

   height  width        other_colors  sellable_online  rating  reviews_count  \
0      87     65  Black, Beige, Blue             True     4.2             21   
1      36    173    Oak, Beige, Blue            False     3.4            306   
2     

In [4]:
# Convert dataset into LangChain Documents

documents = []

for _, row in df.iterrows():
    document = Document(
        page_content=str(row["content"]),
        metadata={
            "item_id": str(row["item_id"]),
            "name": str(row["name"]),
            "category": str(row["category"]),
            "price": str(row["price"])
        }
    )

    documents.append(document)

print("Documents created:", len(documents))

Documents created: 1200


In [5]:
# Split documents into smaller chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 1200


In [6]:
# Load Hugging Face embedding model

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

C:\Users\priyanshu\AppData\Local\Temp\ipykernel_27940\24340380.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [7]:
# Create FAISS vector store

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector store created successfully!")

FAISS vector store created successfully!


In [8]:
# Create vectorstore folder

os.makedirs("../vectorstore/faiss_index", exist_ok=True)

# Save FAISS vector store

vectorstore.save_local(
    "../vectorstore/faiss_index"
)

print("FAISS vector store saved successfully!")
print("Location: vectorstore/faiss_index")

FAISS vector store saved successfully!
Location: vectorstore/faiss_index
